## Notebook 05: Feature Engineering

### Objective
This notebook prepares the data for machine learning by creating new features, encoding categorical variables, scaling numerical features, and splitting the data for training and testing.

### Outcomes
- Create new features from existing data
- Encode categorical variables using one-hot encoding
- Scale numerical features for better model performance
- Split data into training and testing sets
- Handle class imbalance

### Steps Covered
1. Load cleaned dataset
2. Create new features
3. Encode categorical variables
4. Scale numerical features
5. Split data into train/test sets
6. Handle class imbalance with SMOTE
7. Save processed data for modeling

### Importing libraries and loading cleaned dataset

In [1]:
### Importing libraries and loading cleaned dataset

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

CLEANED_DATA_PATH = "../outputs/cleaned_data/bank_cleaned.csv"
PROCESSED_DATA_PATH = "../outputs/cleaned_data/bank_processed.csv"

print("-"*140)
print("LOADING CLEANED DATASET")
print("-"*140)

df = pd.read_csv(CLEANED_DATA_PATH)

print(f"Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))

--------------------------------------------------------------------------------------------------------------------------------------------
LOADING CLEANED DATASET
--------------------------------------------------------------------------------------------------------------------------------------------
Dataset shape: 45211 rows and 18 columns
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y', 'y_binary']

First 3 rows:
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   

    contact  day month  duration  campaign  pdays  previous poutcome   y  \
0  cellular    5   may       261         1     -1         0  failure  no   

### Creating new features


In [2]:
### Creating new features

print("-"*140)
print("CREATING NEW FEATURES")
print("-"*140)

# Create age group feature
df['age_group'] = pd.cut(df['age'], 
                          bins=[0, 25, 35, 45, 55, 65, 100],
                          labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

# Create balance category feature
df['balance_category'] = pd.cut(df['balance'],
                                  bins=[-10000, 0, 500, 2000, 5000, 100000],
                                  labels=['Negative', 'Low (0-500)', 'Medium (500-2k)', 
                                          'High (2k-5k)', 'Very High (5k+)'])

# Create interaction feature: campaign * previous
df['campaign_previous'] = df['campaign'] * df['previous']

# Create flag for multiple contacts
df['multiple_contacts'] = (df['campaign'] > 2).astype(int)

# Create flag for high balance
df['high_balance'] = (df['balance'] > 2000).astype(int)

print("New features created:")
print(f"  - age_group: {df['age_group'].nunique()} categories")
print(f"  - balance_category: {df['balance_category'].nunique()} categories")
print(f"  - campaign_previous: interaction feature")
print(f"  - multiple_contacts: flag (0/1)")
print(f"  - high_balance: flag (0/1)")

print(f"\nDataset shape after feature creation: {df.shape[0]} rows and {df.shape[1]} columns")

print("\nSample of new features:")
print(df[['age', 'age_group', 'balance', 'balance_category', 'campaign', 
          'previous', 'campaign_previous', 'multiple_contacts', 'high_balance']].head(5))

--------------------------------------------------------------------------------------------------------------------------------------------
CREATING NEW FEATURES
--------------------------------------------------------------------------------------------------------------------------------------------
New features created:
  - age_group: 6 categories
  - balance_category: 5 categories
  - campaign_previous: interaction feature
  - multiple_contacts: flag (0/1)
  - high_balance: flag (0/1)

Dataset shape after feature creation: 45211 rows and 23 columns

Sample of new features:
   age age_group  balance balance_category  campaign  previous  \
0   58     56-65     2143     High (2k-5k)         1         0   
1   44     36-45       29      Low (0-500)         1         0   
2   33     26-35        2      Low (0-500)         1         0   
3   47     46-55     1506  Medium (500-2k)         1         0   
4   33     26-35        1      Low (0-500)         1         0   

   campaign_previo

### Selecting features for modeling

In [3]:
### Selecting features for modeling

print("-"*140)
print("SELECTING FEATURES FOR MODELING")
print("-"*140)

# Define feature columns to use for modeling
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 
                        'loan', 'contact', 'month', 'poutcome', 'age_group', 
                        'balance_category']

numerical_features = ['age', 'balance', 'duration', 'campaign', 'pdays', 
                      'previous', 'campaign_previous', 'multiple_contacts', 
                      'high_balance']

# Define target column
target_column = 'y_binary'

print(f"Categorical features ({len(categorical_features)}):")
for feat in categorical_features:
    print(f"  - {feat}")

print(f"\nNumerical features ({len(numerical_features)}):")
for feat in numerical_features:
    print(f"  - {feat}")

print(f"\nTarget column: {target_column}")

# Create X and y
X = df[categorical_features + numerical_features]
y = df[target_column]

print(f"\nFeatures shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print(f"Positive class percentage: {y.mean() * 100:.2f}%")

--------------------------------------------------------------------------------------------------------------------------------------------
SELECTING FEATURES FOR MODELING
--------------------------------------------------------------------------------------------------------------------------------------------
Categorical features (11):
  - job
  - marital
  - education
  - default
  - housing
  - loan
  - contact
  - month
  - poutcome
  - age_group
  - balance_category

Numerical features (9):
  - age
  - balance
  - duration
  - campaign
  - pdays
  - previous
  - campaign_previous
  - multiple_contacts
  - high_balance

Target column: y_binary

Features shape (X): (45211, 20)
Target shape (y): (45211,)
Positive class percentage: 11.70%


### Encoding categorical features and scaling numerical features

In [4]:
### Encoding categorical features and scaling numerical features

print("-"*140)
print("ENCODING CATEGORICAL FEATURES AND SCALING NUMERICAL FEATURES")
print("-"*140)

# Create preprocessor that applies different rules to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),  # Rule for numbers: scale them
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)  # Rule for words: convert to 0/1
    ])

# Apply the rules to the data
X_processed = preprocessor.fit_transform(X)

# Get the names of all new columns created
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = list(numerical_features) + list(cat_feature_names)

print(f"Original features: {X.shape[1]}")
print(f"Features after encoding and scaling: {X_processed.shape[1]}")
print(f"\nFirst 5 feature names:")
print(all_feature_names[:10])

print(f"\nPreprocessor created successfully.")

--------------------------------------------------------------------------------------------------------------------------------------------
ENCODING CATEGORICAL FEATURES AND SCALING NUMERICAL FEATURES
--------------------------------------------------------------------------------------------------------------------------------------------
Original features: 20
Features after encoding and scaling: 50

First 5 feature names:
['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'campaign_previous', 'multiple_contacts', 'high_balance', 'job_blue-collar']

Preprocessor created successfully.


### Splitting data and handling class imbalance with SMOTE

In [5]:
### Splitting data and handling class imbalance with SMOTE

print("-"*140)
print("SPLITTING DATA AND HANDLING CLASS IMBALANCE WITH SMOTE")
print("-"*140)

# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Before SMOTE - Training set:")
print(f"  Class 0 (no):  {(y_train == 0).sum()}")
print(f"  Class 1 (yes): {(y_train == 1).sum()}")
print(f"  Imbalance ratio: {(y_train == 0).sum() / (y_train == 1).sum():.2f}")

print(f"\nBefore SMOTE - Testing set:")
print(f"  Class 0 (no):  {(y_test == 0).sum()}")
print(f"  Class 1 (yes): {(y_test == 1).sum()}")

# Apply SMOTE to balance the training data only
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE - Training set (balanced):")
print(f"  Class 0 (no):  {(y_train_balanced == 0).sum()}")
print(f"  Class 1 (yes): {(y_train_balanced == 1).sum()}")
print(f"  Balance ratio: 1:1")

print(f"\nTraining set shape after SMOTE: {X_train_balanced.shape}")
print(f"Testing set shape (unchanged): {X_test.shape}")

print("\nSMOTE applied successfully. Training data is now balanced.")

--------------------------------------------------------------------------------------------------------------------------------------------
SPLITTING DATA AND HANDLING CLASS IMBALANCE WITH SMOTE
--------------------------------------------------------------------------------------------------------------------------------------------
Before SMOTE - Training set:
  Class 0 (no):  31937
  Class 1 (yes): 4231
  Imbalance ratio: 7.55

Before SMOTE - Testing set:
  Class 0 (no):  7985
  Class 1 (yes): 1058

After SMOTE - Training set (balanced):
  Class 0 (no):  31937
  Class 1 (yes): 31937
  Balance ratio: 1:1

Training set shape after SMOTE: (63874, 50)
Testing set shape (unchanged): (9043, 50)

SMOTE applied successfully. Training data is now balanced.


### Saving processed data and preprocessor for modeling

In [6]:
### Saving processed data and preprocessor for modeling

import joblib

print("-"*140)
print("SAVING PROCESSED DATA AND PREPROCESSOR")
print("-"*140)

# Create directory if it doesn't exist
os.makedirs("../outputs/cleaned_data", exist_ok=True)
os.makedirs("../outputs/models", exist_ok=True)

# Save the processed arrays
np.save("../outputs/cleaned_data/X_train_balanced.npy", X_train_balanced)
np.save("../outputs/cleaned_data/y_train_balanced.npy", y_train_balanced)
np.save("../outputs/cleaned_data/X_test.npy", X_test)
np.save("../outputs/cleaned_data/y_test.npy", y_test)

# Save the preprocessor for later use
joblib.dump(preprocessor, "../outputs/models/preprocessor.pkl")

print("Files saved successfully:")
print(f"  - ../outputs/cleaned_data/X_train_balanced.npy ({X_train_balanced.shape})")
print(f"  - ../outputs/cleaned_data/y_train_balanced.npy ({y_train_balanced.shape})")
print(f"  - ../outputs/cleaned_data/X_test.npy ({X_test.shape})")
print(f"  - ../outputs/cleaned_data/y_test.npy ({y_test.shape})")
print(f"  - ../outputs/models/preprocessor.pkl")

print("\n" + "-"*140)
print("NOTEBOOK 05 COMPLETED SUCCESSFULLY")
print("-"*140)
print("\nProceed to Notebook 06 for model training.")

--------------------------------------------------------------------------------------------------------------------------------------------
SAVING PROCESSED DATA AND PREPROCESSOR
--------------------------------------------------------------------------------------------------------------------------------------------
Files saved successfully:
  - ../outputs/cleaned_data/X_train_balanced.npy ((63874, 50))
  - ../outputs/cleaned_data/y_train_balanced.npy ((63874,))
  - ../outputs/cleaned_data/X_test.npy ((9043, 50))
  - ../outputs/cleaned_data/y_test.npy ((9043,))
  - ../outputs/models/preprocessor.pkl

--------------------------------------------------------------------------------------------------------------------------------------------
NOTEBOOK 05 COMPLETED SUCCESSFULLY
--------------------------------------------------------------------------------------------------------------------------------------------

Proceed to Notebook 06 for model training.
